# Pipeline

In [255]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import random
from sklearn.metrics import roc_auc_score, roc_curve

## Parameters

In [256]:
network = nx.watts_strogatz_graph(300, 4, 0.3) # Network
beta = 0.1 # Infection rate
gamma = 0.1 # Recovery rate
initial_infected_count = 50 # Number of initial infected nodes
asymptomatic_rate = 0.25 # Fraction of asymptomatic nodes

## Defining asymptomatics

In [257]:
def choose_asymptomatic_nodes(graph, fraction, seed=None):
    random.seed(seed)
    nodes = list(graph.nodes())
    k = int(len(nodes) * fraction)
    return set(random.sample(nodes, k))

In [258]:
asymptomatic = choose_asymptomatic_nodes(network, fraction = asymptomatic_rate)
len(asymptomatic), asymptomatic

(75,
 {2,
  6,
  8,
  9,
  16,
  22,
  30,
  34,
  41,
  48,
  51,
  59,
  69,
  83,
  86,
  89,
  90,
  95,
  98,
  102,
  103,
  112,
  113,
  122,
  124,
  129,
  131,
  135,
  136,
  138,
  139,
  146,
  158,
  161,
  166,
  168,
  169,
  170,
  173,
  178,
  179,
  185,
  193,
  194,
  196,
  198,
  202,
  206,
  207,
  208,
  214,
  217,
  220,
  224,
  226,
  227,
  228,
  231,
  233,
  242,
  245,
  248,
  250,
  254,
  259,
  269,
  274,
  275,
  276,
  281,
  286,
  287,
  289,
  297,
  299})

## SIS epidemic

In [259]:
def epidemic_step(graph, beta, gamma, infected):

    new_infected = set()
    recovered = set()

    for node in infected:
        for neighbor in set(graph.neighbors(node)) - infected:
            if random.random() < beta:
                new_infected.add(neighbor)

    for node in infected:
        if random.random() < gamma:
            recovered.add(node)

    infected = infected | new_infected
    infected = infected - recovered

    return infected


def burn_in(graph, beta, gamma, initial_infected_count, burn_in_steps):
    initial_infected = random.sample(list(graph.nodes()), k=initial_infected_count) # Randomly select the initial infected nodes from the graph
    infected = set(initial_infected)

    for _ in range(burn_in_steps):
        infected = epidemic_step(graph, beta, gamma, infected)
    
    return infected

In [260]:
infected = burn_in(network, beta, gamma, initial_infected_count, 1000)
len(infected), infected

(194,
 {0,
  1,
  2,
  4,
  6,
  12,
  14,
  16,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  27,
  28,
  29,
  30,
  32,
  33,
  35,
  40,
  46,
  47,
  48,
  50,
  53,
  54,
  55,
  56,
  57,
  58,
  59,
  61,
  62,
  64,
  65,
  66,
  68,
  69,
  70,
  71,
  72,
  73,
  77,
  78,
  79,
  81,
  82,
  83,
  84,
  85,
  86,
  87,
  89,
  90,
  91,
  92,
  93,
  94,
  95,
  98,
  100,
  101,
  103,
  106,
  108,
  109,
  110,
  112,
  113,
  116,
  118,
  121,
  123,
  125,
  126,
  128,
  129,
  132,
  134,
  137,
  139,
  140,
  141,
  142,
  144,
  145,
  146,
  147,
  148,
  152,
  153,
  154,
  155,
  156,
  157,
  159,
  160,
  161,
  162,
  163,
  164,
  165,
  166,
  167,
  168,
  170,
  172,
  173,
  174,
  175,
  176,
  177,
  178,
  179,
  180,
  181,
  182,
  183,
  186,
  187,
  189,
  192,
  195,
  197,
  198,
  199,
  201,
  202,
  203,
  204,
  206,
  207,
  209,
  210,
  211,
  212,
  213,
  214,
  216,
  217,
  219,
  222,
  223,
  229,
  232,
  233,
  234,
  235,

## Collecting snapshots

In [261]:
def collect_snapshots(graph, beta, gamma, infected, num_snapshots, step_between):
    snapshots = []

    for _ in range(num_snapshots):

        for _ in range(step_between):
            infected = epidemic_step(graph, beta, gamma, infected)

        snapshots.append(set(infected))

    return snapshots

Salva o conjunto de todos os infectados em cada snapshot

In [262]:
snapshots = collect_snapshots(network, beta, gamma, infected, 10, 2)
for i in snapshots:
    print(len(i))

195
204
209
205
200
191
190
199
196
190


## Observed betweenness

In [263]:
def observed_betweenness(graph, observed_nodes):
    """
    Compute the betweenness centrality of nodes considering only
    shortest paths that have both source and target nodes in the
    observed infected nodes set.

    Parameters:
    - graph: NetworkX graph
    - observed_nodes: List of observed infected nodes

    Returns:
    - A dictionary with the betweenness centrality scores.
    """

    return nx.betweenness_centrality_subset(graph, sources=observed_nodes, targets=observed_nodes, normalized=False)

In [264]:
betws = []
observations = {}

for k, I_t in enumerate(snapshots):
    O_t = (I_t - asymptomatic)
    obs_betw = observed_betweenness(network, O_t)
    betws.append(obs_betw)
    if k > 0:
        O_t = O_t | observations[k-1]
    observations[k] = O_t

In [265]:
for O_T in observations.values():
    print("Number of candidates: " + str(len(network.nodes()) - len(O_T)))

Number of candidates: 152
Number of candidates: 125
Number of candidates: 102
Number of candidates: 93
Number of candidates: 89
Number of candidates: 85
Number of candidates: 83
Number of candidates: 81
Number of candidates: 81
Number of candidates: 81


## Evaluating

In [266]:
def auc_score(asymptomatics, centrality_measures, observed_nodes):
    """
    Calculate the AUC score and ROC curve, given the set of infected nodes,
    the network centrality measure used for ranking, and a list of observed 
    nodes to be excluded from the calculation.

    Parameters:
    - infected_nodes: List of infected nodes
    - centrality_measures: Dictionary where the keys are node indices and
                           the values are their corresponding centrality
                           measures (e.g., degree, betweenness, etc.)
    - observed_nodes : List of node indices to exclude from evaluation
    
    Returns:
    - auc: The Area Under the Curve (AUC) score
    - curve: Tuple containing the False Positive Rate (FPR),
             True Positive Rate (TPR), and thresholds for
             plotting the ROC curve
    """
    # Filter out observed nodes
    evaluation_nodes = [node for node in centrality_measures if node not in observed_nodes]

    # Build ground truth and scores
    y_true = []
    y_scores = []

    for node in evaluation_nodes:
        y_true.append(1 if node in asymptomatics else 0)
        y_scores.append(centrality_measures[node])

    # Compute the AUC score
    auc = roc_auc_score(y_true, y_scores)
    # Compute the ROC curve
    curve = roc_curve(y_true, y_scores)

    return {'auc': auc, 'roc': curve}

In [267]:
AUCS = []
FPS = []
FPS_BETW = []
top = 75

for k, s in enumerate(snapshots):
    print("iteration", k)

    betw = {i: sum(d[i] for d in betws[:k+1]) for i in betws[0]}
    # betw = betws[k]
    # obs = set().union(*(observations[i] for i in range(k+1)))
    obs = observations[k]
    AUC = auc_score(asymptomatic, betw, obs)['auc'] # Check if this makes sense
    AUCS.append(AUC)

    evaluation_nodes = [node for node in betw if node not in obs]
    betw_eval = {}
    for i in evaluation_nodes:
        betw_eval[i] = betw[i]
    betw_rank = sorted_data_desc = {k: v for k, v in sorted(betw_eval.items(), key=lambda item: item[1], reverse=True)}
    top_k = list(betw_rank.keys())[:top]

    FP_BETW = len((set(top_k) - asymptomatic)) / len(network.nodes() - obs)
    FPS_BETW.append(FP_BETW)


    FP = len((network.nodes() - obs) - asymptomatic)  / len(network.nodes() - obs)
    FPS.append(FP)


print("AUCS:", AUCS)
print("FPS:", FPS)
print("FPS_BETW:", FPS_BETW)

iteration 0
iteration 1
iteration 2
iteration 3
iteration 4
iteration 5
iteration 6
iteration 7
iteration 8
iteration 9
AUCS: [np.float64(0.6000865800865801), np.float64(0.6689333333333334), np.float64(0.7308641975308643), np.float64(0.8066666666666666), np.float64(0.8076190476190476), np.float64(0.7946666666666666), np.float64(0.8366666666666667), np.float64(0.8022222222222222), np.float64(0.8044444444444445), np.float64(0.8066666666666666)]
FPS: [0.506578947368421, 0.4, 0.2647058823529412, 0.1935483870967742, 0.15730337078651685, 0.11764705882352941, 0.0963855421686747, 0.07407407407407407, 0.07407407407407407, 0.07407407407407407]
FPS_BETW: [0.21052631578947367, 0.184, 0.14705882352941177, 0.10752688172043011, 0.0898876404494382, 0.058823529411764705, 0.04819277108433735, 0.04938271604938271, 0.04938271604938271, 0.04938271604938271]
